In [1]:
import os
from pathlib import Path
import uuid
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb

c:\Users\MYCOMPUTER\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_text_files(folder):
    folder = Path(folder)
    text_files = [p for p in folder.glob('*.txt') if p.is_file()]
    print(f'Found {len(text_files)} text files in {folder}')

    documents = []
    for path in text_files:
        text = path.read_text(encoding='utf-8')
        documents.append({
            'source_file': path.name,
            'text': text,
            'source_path': str(path),
        })
        print(f'  Loaded {path.name} ({len(text)} chars)')
    return documents

documents = load_text_files('text_files')

Found 6 text files in text_files
  Loaded current_wc_text.txt (41742 chars)
  Loaded filgoal_matches.txt (346 chars)
  Loaded filgoal_news.txt (5882 chars)
  Loaded google.txt (7515 chars)
  Loaded worldcup_history_text.txt (36464 chars)
  Loaded world_cup_matches.txt (346 chars)


## 2. chunks

In [3]:
def split_text(text, chunk_size=300, chunk_overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - chunk_overlap
    return chunks

all_chunks = []
for doc in documents:
    chunks = split_text(doc['text'], chunk_size=300, chunk_overlap=30)
    for chunk in chunks:
        all_chunks.append({
            'source_file': doc['source_file'],
            'text': chunk,
        })
print(f'Total chunks: {len(all_chunks)}')

Total chunks: 345


## 3. Embeddings

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')
texts = [chunk['text'] for chunk in all_chunks]
print(f'Generating embeddings for {len(texts)} chunks...')
embeddings = model.encode(texts, show_progress_bar=True)
print('Embeddings generated. Shape:', embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6852.57it/s]


Generating embeddings for 345 chunks...


Batches: 100%|██████████| 11/11 [00:07<00:00,  1.49it/s]

Embeddings generated. Shape: (345, 384)


## 4. Store

In [5]:
store_path = 'data/vector_store'
os.makedirs(store_path, exist_ok=True)
client = chromadb.PersistentClient(path=store_path)
collection = client.get_or_create_collection(
    name='pdf_documents',
    metadata={
        'description': 'World Cup text embeddings',
        'hnsw:space': 'cosine'
    }
)
print('Collection ready. Current count:', collection.count())

ids = [f'doc_{uuid.uuid4().hex[:8]}_{i}' for i in range(len(all_chunks))]
metadatas = [{'source_file': chunk['source_file'], 'content_length': len(chunk['text'])} for chunk in all_chunks]
documents_text = [chunk['text'] for chunk in all_chunks]
normalized_embeddings = []
for vector in embeddings:
    arr = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(arr)
    if norm > 0:
        arr = arr / norm
    normalized_embeddings.append(arr.tolist())

collection.add(
    ids=ids,
    embeddings=normalized_embeddings,
    documents=documents_text,
    metadatas=metadatas,
)
print('Stored', len(ids), 'chunks in the vector store.')
print('Final count:', collection.count())

Collection ready. Current count: 0
Stored 345 chunks in the vector store.
Final count: 345
